In [2]:
import requests
import xml.etree.ElementTree as ET

def fetch_and_parse_urlset(url):
    try:
        print(f"🔍 Fetching sitemap: {url}")
        response = requests.get(url)
        response.raise_for_status()

        root = ET.fromstring(response.content)
        namespace = {'ns': 'http://www.sitemaps.org/schemas/sitemap/0.9'}
        url_tags = root.findall('ns:url', namespace)

        print(f"\n📰 Found {len(url_tags)} article URLs:\n")

        for idx, url_tag in enumerate(url_tags, 1):
            loc = url_tag.find('ns:loc', namespace).text
            lastmod = url_tag.find('ns:lastmod', namespace).text if url_tag.find('ns:lastmod', namespace) is not None else "N/A"
            changefreq = url_tag.find('ns:changefreq', namespace).text if url_tag.find('ns:changefreq', namespace) is not None else "N/A"
            priority = url_tag.find('ns:priority', namespace).text if url_tag.find('ns:priority', namespace) is not None else "N/A"

            print(f"{idx}. URL           : {loc}")
            print(f"   Last Modified : {lastmod}")
            print(f"   Change Freq   : {changefreq}")
            print(f"   Priority      : {priority}\n")

    except requests.RequestException as e:
        print(f"❌ Request failed: {e}")
    except ET.ParseError as e:
        print(f"❌ XML parsing failed: {e}")

if __name__ == "__main__":
    updated_sitemap_url = "https://www.glass-international.com/sitemaps-1-section-news-1-sitemap.xml"  # Replace with your exact sitemap URL
    fetch_and_parse_urlset(updated_sitemap_url)


🔍 Fetching sitemap: https://www.glass-international.com/sitemaps-1-section-news-1-sitemap.xml

📰 Found 6820 article URLs:

1. URL           : https://www.glass-international.com/news/sorg-delivers-f6-furnace-for-vetri-speciali
   Last Modified : 2025-07-21T11:19:22+00:00
   Change Freq   : weekly
   Priority      : 0.5

2. URL           : https://www.glass-international.com/news/fives-joins-la-glass-vallee
   Last Modified : 2025-07-18T09:11:31+00:00
   Change Freq   : weekly
   Priority      : 0.5

3. URL           : https://www.glass-international.com/news/ardagh-to-produce-low-carbon-glass-bottles-for-jagermeister
   Last Modified : 2025-07-17T11:26:33+00:00
   Change Freq   : weekly
   Priority      : 0.5

4. URL           : https://www.glass-international.com/news/qemetica-invests-pln-70-million-in-polish-glass-facility
   Last Modified : 2025-07-16T09:14:00+00:00
   Change Freq   : weekly
   Priority      : 0.5

5. URL           : https://www.glass-international.com/news/uk-medic

In [10]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime
import re

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3',
}

COOKIES = {
    'name_of_the_consent_cookie': 'value_indicating_consent',
}

def try_parse_date(date_str):
    # Remove ordinal suffixes (like 'st', 'nd', 'rd', 'th') before parsing
    date_str = re.sub(r'(\d{1,2})(st|nd|rd|th)', r'\1', date_str)

    formats = [
        "%B %d, %Y",   # e.g., September 1, 2023
        "%d %B %Y",    # e.g., 1 September 2023
        "%Y-%m-%d",    # ISO format
        "%d. %B %Y",   # e.g., 1. September 2023
        "%d %B, %Y",   # e.g., 16 July, 2025
    ]

    for fmt in formats:
        try:
            return datetime.strptime(date_str, fmt).strftime("%Y-%m-%d")
        except Exception:
            continue
    return date_str

def parse_article(url):
    print(f"\n[🌐] Scraping: {url}")
    try:
        response = requests.get(url, headers=HEADERS, timeout=30)
        response.raise_for_status()
    except Exception as e:
        print(f"[!] Failed to fetch article: {e}")
        return

    soup = BeautifulSoup(response.content, "html.parser")

    # Title
    title_tag = soup.select_one("h1.mb-2 pb-2 text-2xl border-b border-grey-light")
    title = title_tag.get_text(strip=True) if title_tag else "No title found"

    # Excerpt (optional)
    excerpt_tag = soup.select_one("p.article-header__excerpt")
    excerpt = excerpt_tag.get_text(strip=True) if excerpt_tag else None

    # Date
    published_tag = soup.select_one("div.publishedby p")
    text = published_tag.get_text(strip=True)
    match = re.search(r"Published (\d{1,2}(?:st|nd|rd|th)? [A-Za-z]+, \d{4})", text)
    date_raw = match.group(1) if match else "No date found"
    print("Raw date:", date_raw)
    date = try_parse_date(date_raw)


    # Paragraphs
    paragraphs = [p.get_text(strip=True) for p in soup.select("p") if p.get_text(strip=True)]

    # Print results
    print(f"📰 Title: {title}")
    if excerpt:
        print(f"💡 Excerpt: {excerpt}")
    print(f"📅 Published: {date}")
    print(f"📄 Paragraphs ({len(paragraphs)}):")
    for i, para in enumerate(paragraphs, 1):
        print(f"  {i}. {para}")

if __name__ == "__main__":
    urls = [
        "https://www.glass-international.com/news/british-glass-announces-structural-changes"
    ]

    for url in urls:
        parse_article(url)


[🌐] Scraping: https://www.glass-international.com/news/gerresheimer-participates-in-furnace-for-the-future-project
Raw date: 15th April, 2021
📰 Title: No title found
📅 Published: 2021-04-15
📄 Paragraphs (21):
  1. Published 15th April, 2021 by Greg Morris
  2. Gerresheimeris one of a total of 19 glass producers who have joined forces in a project with the aim of achieving climate-neutral glass production.
  3. In collaboration with the Ardagh Group, the group wants to drive forward the development, financing and operation of a hybrid electric melting furnace.
  4. An industrial-scale furnace is being built in Obernkirchen, Germany, for the commercial production of glass containers from renewable electricity.
  5. The demonstration facility will be built in 2022, with initial results available in 2023.
  6. Both technical and market-specific criteria for melting glass to produce glass packaging on a large scale will be evaluated.
  7. The companies participating in the Furnace for the 